<a href="https://colab.research.google.com/github/ricardoalvarezv-cpu/Memoria-LSTM-SSI/blob/main/LSTM_Teacher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM_Teacher — Entrenamiento y Evaluación

Este notebook implementa el entrenamiento y evaluación del modelo **LSTM_Teacher**,
el cual aproxima la respuesta dinámica del suelo en un problema de interacción
suelo–estructura.

El modelo recibe como entrada:

vg, ag, d_mid_x, v_mid_x, a_mid_x

y predice:

a_base_x, Fx, Mz

El entrenamiento final del modelo se realiza utilizando **ventanas temporales
solapadas**, estrategia que mostró el mejor desempeño frente a otras alternativas.

Además se incluye una sección metodológica donde se comparan tres estrategias:

A. Entrenamiento por ventanas  
B. Entrenamiento por secuencias completas  
C. Ventanas + fine tuning por evento completo

Los resultados de esta comparación justifican la elección del modelo final.

# 1.- Preparación del entorno

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 1
# Setup general: GPU, paths, dataset y parámetros globales
# ============================================
import os
import glob
import numpy as np
import torch
from pathlib import Path

# ----------------------------
# Dispositivo de cómputo
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

# ----------------------------
# Paths principales
# ----------------------------
ROOT = "/content/memoria"

# Detecta processed_step_full_v2 sin duplicados
CANDIDATES = [
    "/content/memoria/processed_step_full_v2",
    "/content/memoria/processed_step_full_v2/processed_step_full_v2",
]

PROCESSED_DIR = None
for c in CANDIDATES:
    if os.path.isdir(c) and len(glob.glob(os.path.join(c, "EQ*.npz"))) > 0:
        PROCESSED_DIR = c
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "No encuentro processed_step_full_v2 con EQ*.npz. ¿Montaste bien el volumen?"
    )

ALL_FILES = sorted(glob.glob(os.path.join(PROCESSED_DIR, "EQ*.npz")))

# Output SIEMPRE a carpeta montada en PC
RUNS_DIR_B = os.path.join(ROOT, "runs_colab", "runs_notebook_B_teacher")
os.makedirs(RUNS_DIR_B, exist_ok=True)

print("✅ PROCESSED_DIR:", PROCESSED_DIR)
print("✅ EQ encontrados:", len(ALL_FILES), "| ejemplo:", [os.path.basename(p) for p in ALL_FILES[:5]])
print("✅ RUNS_DIR_B:", RUNS_DIR_B)

# ----------------------------
# Nombres de variables
# ----------------------------
X_NAMES_FULL = ["vg", "ag", "d_mid_x", "v_mid_x", "a_mid_x"]
Y_NAMES      = ["a_base_x", "Fx", "Mz"]

# ----------------------------
# Parámetros globales del problema
# ----------------------------
DT_REF   = 0.005
T_BACK_S = 5.12
T_BACK   = int(round(T_BACK_S / DT_REF))   # 1024
EPS      = 1e-8

# clipping global recomendado
CLIP_X = 10.0
CLIP_Y = 10.0

print("✅ DT_REF:", DT_REF)
print("✅ T_BACK_S:", T_BACK_S)
print("✅ T_BACK:", T_BACK)

DEVICE: cuda
GPU: NVIDIA GeForce RTX 4060
✅ PROCESSED_DIR: /content/memoria/processed_step_full_v2
✅ EQ encontrados: 84 | ejemplo: ['EQ001.npz', 'EQ002.npz', 'EQ003.npz', 'EQ004.npz', 'EQ005.npz']
✅ RUNS_DIR_B: /content/memoria/runs_colab/runs_notebook_B_teacher
✅ DT_REF: 0.005
✅ T_BACK_S: 5.12
✅ T_BACK: 1024


In [ ]:
import torch, os

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    try:
        print("device 0:", torch.cuda.get_device_name(0))
        x = torch.zeros(1, device="cuda")
        print("mini test cuda OK:", x)
    except Exception as e:
        print("mini test cuda FAIL:", repr(e))

torch: 2.9.0+cu126
cuda available: True
cuda device count: 1
device 0: NVIDIA GeForce RTX 4060
mini test cuda OK: tensor([0.], device='cuda:0')


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 2
# Funciones base del pipeline:
#   - carga de eventos .npz
#   - resampling a dt_ref
#   - modelo LSTM Teacher
#   - funciones de pérdida
# ============================================
import os
import numpy as np
import torch
import torch.nn as nn

# ----------------------------
# Loader de eventos .npz
# ----------------------------
def load_event_npz(path: str):
    z = np.load(path, allow_pickle=True)
    keys = list(z.files)

    if "X_full" not in keys or "Y_full" not in keys:
        raise KeyError(f"Faltan X_full/Y_full en {path}. Keys={keys}")

    X = z["X_full"].astype(np.float32)   # (NT, 5)
    Y = z["Y_full"].astype(np.float32)   # (NT, 3)

    dt = float(np.array(z["dt"]).reshape(())) if "dt" in keys else None
    rid = (
        str(np.array(z["record_id"]).reshape(()))
        if "record_id" in keys
        else os.path.splitext(os.path.basename(path))[0]
    )

    assert X.ndim == 2 and X.shape[1] == 5, (rid, X.shape)
    assert Y.ndim == 2 and Y.shape[1] == 3, (rid, Y.shape)

    if dt is None:
        raise ValueError(f"{rid}: no viene dt en el npz.")

    return rid, X, Y, dt

# ----------------------------
# Helper de suavizado simple
# ----------------------------
def _moving_average(x, w):
    if w <= 1:
        return x
    k = np.ones(w, dtype=np.float64) / w
    y = np.empty_like(x, dtype=np.float64)
    for j in range(x.shape[1]):
        y[:, j] = np.convolve(x[:, j], k, mode="same")
    return y.astype(np.float32)

# ----------------------------
# Resampling a dt_ref
# ----------------------------
def resample_to_dt_ref(X, Y, dt, dt_ref=DT_REF, smooth_if_downsample=True):
    """
    Resamplea X e Y a una grilla temporal uniforme con paso dt_ref.
    Si dt_ref > dt, aplica un suavizado simple previo para reducir aliasing.
    """
    NT = X.shape[0]
    t = np.arange(NT, dtype=np.float64) * float(dt)
    t_end = t[-1]

    # grilla temporal nueva robusta a errores de floating point
    t2 = np.arange(0.0, t_end + 0.5 * dt_ref, float(dt_ref), dtype=np.float64)
    NT2 = t2.size

    # downsample -> suavizado simple previo
    if smooth_if_downsample and (dt_ref > dt * 1.01):
        ratio = dt_ref / dt
        w = int(np.ceil(ratio))
        Xs = _moving_average(X, w)
        Ys = _moving_average(Y, w)
    else:
        Xs, Ys = X, Y

    Xr = np.empty((NT2, X.shape[1]), dtype=np.float32)
    Yr = np.empty((NT2, Y.shape[1]), dtype=np.float32)

    for j in range(X.shape[1]):
        Xr[:, j] = np.interp(t2, t, Xs[:, j]).astype(np.float32)
    for j in range(Y.shape[1]):
        Yr[:, j] = np.interp(t2, t, Ys[:, j]).astype(np.float32)

    return Xr, Yr, float(dt_ref)

# ----------------------------
# Modelo LSTM Teacher
# ----------------------------
class TeacherLSTM(nn.Module):
    def __init__(self, in_dim=5, hid=256, layers=2, dropout=0.0, out_dim=3):
        super().__init__()
        do = float(dropout) if layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hid,
            num_layers=layers,
            batch_first=True,
            dropout=do
        )
        self.head = nn.Linear(hid, out_dim)

    def forward(self, x):
        h, _ = self.lstm(x)
        y = self.head(h)
        return y

# ----------------------------
# Funciones de pérdida
# ----------------------------
def mse_loss(yhat, y):
    return ((yhat - y) ** 2).mean()

def scale_loss_global(yhat, y, eps=1e-6):
    """
    Penaliza diferencia global de escala entre predicción y verdad.
    """
    std_t = y.std(dim=(0, 1), unbiased=False)
    std_p = yhat.std(dim=(0, 1), unbiased=False)
    ratio = std_p / (std_t + eps)
    return ((ratio - 1.0) ** 2).mean()

def bias_loss_weighted(yhat, y, w=(0.0, 0.0, 1.0)):
    """
    Penaliza diferencia de medias por canal.
    Por defecto castiga solo el sesgo en Mz.
    """
    w = torch.tensor(w, device=yhat.device, dtype=yhat.dtype).view(1, 1, -1)
    mu_t = y.mean(dim=(0, 1), keepdim=True)
    mu_p = yhat.mean(dim=(0, 1), keepdim=True)
    return (((mu_p - mu_t) * w) ** 2).mean()

# ----------------------------
# Smoke tests
# ----------------------------
rid, X, Y, dt = load_event_npz(ALL_FILES[0])
Xr, Yr, dt2 = resample_to_dt_ref(X, Y, dt)

print("✅ Smoke test loader/resample:")
print("   rid:", rid)
print("   X original:", X.shape, "| Y original:", Y.shape, "| dt original:", dt)
print("   X resampled:", Xr.shape, "| Y resampled:", Yr.shape, "| dt_ref:", dt2)

model_smoke = TeacherLSTM(in_dim=5, hid=256, layers=2, dropout=0.0, out_dim=3).to(DEVICE)
xb_smoke = torch.zeros((2, T_BACK, 5), dtype=torch.float32).to(DEVICE)
yb_smoke = model_smoke(xb_smoke)
print("✅ Smoke test modelo:", yb_smoke.shape)

✅ Smoke test loader/resample:
   rid: EQ001
   X original: (7624, 5) | Y original: (7624, 3) | dt original: 0.005
   X resampled: (7624, 5) | Y resampled: (7624, 3) | dt_ref: 0.005
✅ Smoke test modelo: torch.Size([2, 1024, 3])


# 2.- GroupKFoldsetup

In [ ]:
# ============================================
# NOTEBOOK B — NUEVA CELDA 3
# GroupKFold setup para validación metodológica del Teacher
# - 4 eventos holdout finales fuera de TODO
# - GroupKFold solo sobre el pool restante
# - split por EVENTO (sin mezclar ventanas entre train y val)
# ============================================
import os
import numpy as np
from sklearn.model_selection import GroupKFold

def record_id_from_path(p):
    return os.path.splitext(os.path.basename(p))[0]

ALL_IDS = sorted([record_id_from_path(p) for p in ALL_FILES])
print("✅ Total eventos disponibles:", len(ALL_IDS))

# --------------------------------------------------
# Holdout final oficial (4 eventos fuera de TODO)
# --------------------------------------------------
TEST_IDS_FINAL = ["EQ007", "EQ019", "EQ052", "EQ081"]

for tid in TEST_IDS_FINAL:
    assert tid in ALL_IDS, f"{tid} no está en el dataset."

# Pool para CV
cv_pool_files = [
    p for p in ALL_FILES
    if record_id_from_path(p) not in TEST_IDS_FINAL
]
cv_pool_ids = [record_id_from_path(p) for p in cv_pool_files]

print("✅ Pool para GroupKFold:", len(cv_pool_files), "eventos")
print("✅ Holdout final:", len(TEST_IDS_FINAL), TEST_IDS_FINAL)

# --------------------------------------------------
# GroupKFold
# --------------------------------------------------
N_SPLITS_TEACHER = 5
gkf_teacher = GroupKFold(n_splits=N_SPLITS_TEACHER)

teacher_folds = []
dummy_X = np.zeros(len(cv_pool_files))
groups = np.array(cv_pool_ids)

for fold_idx, (tr_idx, va_idx) in enumerate(gkf_teacher.split(dummy_X, groups=groups), start=1):
    tr_files = [cv_pool_files[i] for i in tr_idx]
    va_files = [cv_pool_files[i] for i in va_idx]

    tr_ids = sorted([record_id_from_path(p) for p in tr_files])
    va_ids = sorted([record_id_from_path(p) for p in va_files])

    assert len(set(tr_ids).intersection(set(va_ids))) == 0, f"Fold {fold_idx}: Train y Val comparten eventos."
    assert len(set(tr_ids).intersection(set(TEST_IDS_FINAL))) == 0, f"Fold {fold_idx}: Train y Test comparten eventos."
    assert len(set(va_ids).intersection(set(TEST_IDS_FINAL))) == 0, f"Fold {fold_idx}: Val y Test comparten eventos."

    teacher_folds.append({
        "fold": fold_idx,
        "train_files": tr_files,
        "val_files": va_files,
        "train_ids": tr_ids,
        "val_ids": va_ids,
    })

print("\n=== RESUMEN GROUPKFOLD TEACHER ===")
for fd in teacher_folds:
    print(
        f"Fold {fd['fold']}: "
        f"train={len(fd['train_ids'])} eventos | "
        f"val={len(fd['val_ids'])} eventos"
    )

print("\n✅ GroupKFold Teacher listo.")

✅ Total eventos disponibles: 84
✅ Pool para GroupKFold: 80 eventos
✅ Holdout final: 4 ['EQ007', 'EQ019', 'EQ052', 'EQ081']

=== RESUMEN GROUPKFOLD TEACHER ===
Fold 1: train=64 eventos | val=16 eventos
Fold 2: train=64 eventos | val=16 eventos
Fold 3: train=64 eventos | val=16 eventos
Fold 4: train=64 eventos | val=16 eventos
Fold 5: train=64 eventos | val=16 eventos

✅ GroupKFold Teacher listo.


# 3.- Entrenamiento Final oficial del Teacher

In [ ]:
# ============================================
# BLOQUE 1 — CELDA 4
# Teacher | Split oficial para Resultados
# - VAL teacher = misma validación usada en Teacher
# - TEST final = 4 OOD oficiales
# ============================================
import os
import numpy as np

def record_id_from_path(p):
    return os.path.splitext(os.path.basename(p))[0]

ALL_IDS = sorted([record_id_from_path(p) for p in ALL_FILES])

# --------------------------------------------------
# HOLDOUTS OFICIALES
# --------------------------------------------------
OOD_RIDS = ["EQ007", "EQ019", "EQ052", "EQ081"]

# --------------------------------------------------
# VALIDACIÓN REAL USADA EN TEACHER
# --------------------------------------------------
VAL_IDS_CMP = [
    'EQ004', 'EQ012', 'EQ014', 'EQ021',
    'EQ022', 'EQ027', 'EQ028', 'EQ030',
    'EQ037', 'EQ045', 'EQ050', 'EQ068',
    'EQ070', 'EQ073', 'EQ075', 'EQ083'
]

# --------------------------------------------------
# TRAIN de referencia = todo excepto VAL y OOD
# --------------------------------------------------
TRAIN_IDS_CMP = sorted([
    rid for rid in ALL_IDS
    if rid not in set(VAL_IDS_CMP) and rid not in set(OOD_RIDS)
])

# --------------------------------------------------
# TEST comparativo/final = OOD oficiales
# --------------------------------------------------
TEST_IDS_CMP = list(OOD_RIDS)

val_files_teacher = [
    p for p in ALL_FILES
    if record_id_from_path(p) in VAL_IDS_CMP
]

test_files_cmp = [
    p for p in ALL_FILES
    if record_id_from_path(p) in TEST_IDS_CMP
]

print("=== SPLIT USADO EN RESULTADOS ===")
print("TRAIN_IDS_CMP:", TRAIN_IDS_CMP)
print("VAL_IDS_CMP:  ", VAL_IDS_CMP)
print("TEST_IDS_CMP: ", TEST_IDS_CMP)

assert len(VAL_IDS_CMP) == 16, f"VAL teacher debe tener 16 eventos, pero tiene {len(VAL_IDS_CMP)}"
assert len(TEST_IDS_CMP) == 4, f"TEST debe tener 4 eventos, pero tiene {len(TEST_IDS_CMP)}"
assert len(val_files_teacher) == 16, f"val_files_teacher debe tener 16 eventos, pero tiene {len(val_files_teacher)}"
assert len(test_files_cmp) == 4, f"test_files_cmp debe tener 4 eventos, pero tiene {len(test_files_cmp)}"

print("\n✅ val_files_teacher:", [record_id_from_path(p) for p in val_files_teacher])
print("✅ test_files_cmp:", [record_id_from_path(p) for p in test_files_cmp])

=== SPLIT USADO EN RESULTADOS ===
TRAIN_IDS_CMP: ['EQ001', 'EQ002', 'EQ003', 'EQ005', 'EQ006', 'EQ008', 'EQ009', 'EQ010', 'EQ011', 'EQ013', 'EQ015', 'EQ016', 'EQ018', 'EQ023', 'EQ024', 'EQ025', 'EQ026', 'EQ029', 'EQ031', 'EQ032', 'EQ033', 'EQ034', 'EQ035', 'EQ036', 'EQ038', 'EQ039', 'EQ040', 'EQ041', 'EQ042', 'EQ043', 'EQ044', 'EQ046', 'EQ047', 'EQ048', 'EQ049', 'EQ051', 'EQ053', 'EQ054', 'EQ055', 'EQ056', 'EQ057', 'EQ058', 'EQ059', 'EQ060', 'EQ061', 'EQ062', 'EQ063', 'EQ064', 'EQ065', 'EQ066', 'EQ067', 'EQ069', 'EQ071', 'EQ072', 'EQ074', 'EQ076', 'EQ077', 'EQ078', 'EQ079', 'EQ080', 'EQ082', 'EQ084', 'EQ085', 'EQ086']
VAL_IDS_CMP:   ['EQ004', 'EQ012', 'EQ014', 'EQ021', 'EQ022', 'EQ027', 'EQ028', 'EQ030', 'EQ037', 'EQ045', 'EQ050', 'EQ068', 'EQ070', 'EQ073', 'EQ075', 'EQ083']
TEST_IDS_CMP:  ['EQ007', 'EQ019', 'EQ052', 'EQ081']

✅ val_files_teacher: ['EQ004', 'EQ012', 'EQ014', 'EQ021', 'EQ022', 'EQ027', 'EQ028', 'EQ030', 'EQ037', 'EQ045', 'EQ050', 'EQ068', 'EQ070', 'EQ073', 'EQ075', 'EQ0

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 5
# Scalers globales + caché de eventos
# - Fit SOLO en TRAIN
# - Val y Test usan scalers de TRAIN
# - Construye train_final_files / val_final_files / test_final_files
#   desde TRAIN_IDS_CMP / VAL_IDS_CMP / TEST_IDS_CMP
# - Caché en RAM para acelerar entrenamiento/evaluación
# ============================================
import numpy as np
import os

P_P99 = 99.0

def record_id_from_path(p):
    return os.path.splitext(os.path.basename(p))[0]

def fit_global_scalers_mean_p99abs(files, dt_ref=DT_REF, p=99.0, max_points_per_event=200_000):
    X_list, Y_list = [], []

    for path in files:
        rid, X, Y, dt = load_event_npz(path)
        Xr, Yr, _ = resample_to_dt_ref(X, Y, dt, dt_ref=dt_ref)

        NT = Xr.shape[0]
        if NT > max_points_per_event:
            idx = np.linspace(0, NT - 1, max_points_per_event).astype(int)
            Xr = Xr[idx]
            Yr = Yr[idx]

        X_list.append(Xr.astype(np.float32))
        Y_list.append(Yr.astype(np.float32))

    X_all = np.concatenate(X_list, axis=0)
    Y_all = np.concatenate(Y_list, axis=0)

    XM = np.mean(X_all, axis=0, keepdims=True).astype(np.float32)
    YM = np.mean(Y_all, axis=0, keepdims=True).astype(np.float32)

    X_P99ABS = np.percentile(np.abs(X_all - XM), p, axis=0, keepdims=True).astype(np.float32)
    Y_P99ABS = np.percentile(np.abs(Y_all - YM), p, axis=0, keepdims=True).astype(np.float32)

    X_P99ABS = np.maximum(X_P99ABS, EPS).astype(np.float32)
    Y_P99ABS = np.maximum(Y_P99ABS, EPS).astype(np.float32)

    return XM, YM, X_P99ABS, Y_P99ABS

def build_event_cache(files, XM, YM, X_S, Y_S, dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y):
    """
    Devuelve un diccionario por evento con:
    - rid
    - Xr, Yr (físicos resampleados)
    - Xn, Yn (normalizados)
    - NT
    """
    cache = {}

    for p in files:
        rid, X, Y, dt = load_event_npz(p)
        Xr, Yr, _ = resample_to_dt_ref(X, Y, dt, dt_ref=dt_ref)

        Xn = (Xr - XM) / (X_S + EPS)
        Yn = (Yr - YM) / (Y_S + EPS)

        Xn = np.clip(Xn, -clip_x, clip_x)
        Yn = np.clip(Yn, -clip_y, clip_y)

        Xr = np.nan_to_num(Xr.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        Yr = np.nan_to_num(Yr.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        Xn = np.nan_to_num(Xn.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        Yn = np.nan_to_num(Yn.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

        cache[p] = {
            "rid": rid,
            "Xr": Xr,
            "Yr": Yr,
            "Xn": Xn,
            "Yn": Yn,
            "NT": int(Xr.shape[0]),
        }

    return cache

# --------------------------------------------------
# Construcción explícita de train/val/test files
# --------------------------------------------------
assert "TRAIN_IDS_CMP" in globals(), "Falta TRAIN_IDS_CMP (corre antes la CELDA 4)."
assert "VAL_IDS_CMP"   in globals(), "Falta VAL_IDS_CMP (corre antes la CELDA 4)."
assert "TEST_IDS_CMP"  in globals(), "Falta TEST_IDS_CMP (corre antes la CELDA 4)."

train_final_files = [
    p for p in ALL_FILES
    if record_id_from_path(p) in TRAIN_IDS_CMP
]

val_final_files = [
    p for p in ALL_FILES
    if record_id_from_path(p) in VAL_IDS_CMP
]

test_final_files = [
    p for p in ALL_FILES
    if record_id_from_path(p) in TEST_IDS_CMP
]

print("✅ train_final_files:", [record_id_from_path(p) for p in train_final_files])
print("✅ val_final_files  :", [record_id_from_path(p) for p in val_final_files])
print("✅ test_final_files :", [record_id_from_path(p) for p in test_final_files])

assert len(train_final_files) == len(TRAIN_IDS_CMP), "Mismatch en train_final_files"
assert len(val_final_files) == len(VAL_IDS_CMP), "Mismatch en val_final_files"
assert len(test_final_files) == len(TEST_IDS_CMP), "Mismatch en test_final_files"

# --------------------------------------------------
# Fit de scalers SOLO con TRAIN
# --------------------------------------------------
XM_FINAL, YM_FINAL, X_P99ABS_FINAL, Y_P99ABS_FINAL = fit_global_scalers_mean_p99abs(
    train_final_files, dt_ref=DT_REF, p=P_P99
)

print("\n✅ Scalers finales listos")
print("XM_FINAL:", XM_FINAL)
print("X_P99ABS_FINAL:", X_P99ABS_FINAL)
print("YM_FINAL:", YM_FINAL)
print("Y_P99ABS_FINAL:", Y_P99ABS_FINAL)

# --------------------------------------------------
# Cachés por grupo
# --------------------------------------------------
cache_train_final = build_event_cache(
    train_final_files, XM_FINAL, YM_FINAL, X_P99ABS_FINAL, Y_P99ABS_FINAL,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

cache_val_final = build_event_cache(
    val_final_files, XM_FINAL, YM_FINAL, X_P99ABS_FINAL, Y_P99ABS_FINAL,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

cache_test_final = build_event_cache(
    test_final_files, XM_FINAL, YM_FINAL, X_P99ABS_FINAL, Y_P99ABS_FINAL,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

print("\n✅ cache_train_final:", len(cache_train_final), "eventos")
print("✅ cache_val_final  :", len(cache_val_final), "eventos")
print("✅ cache_test_final :", len(cache_test_final), "eventos")

✅ train_final_files: ['EQ001', 'EQ002', 'EQ003', 'EQ005', 'EQ006', 'EQ008', 'EQ009', 'EQ010', 'EQ011', 'EQ013', 'EQ015', 'EQ016', 'EQ018', 'EQ023', 'EQ024', 'EQ025', 'EQ026', 'EQ029', 'EQ031', 'EQ032', 'EQ033', 'EQ034', 'EQ035', 'EQ036', 'EQ038', 'EQ039', 'EQ040', 'EQ041', 'EQ042', 'EQ043', 'EQ044', 'EQ046', 'EQ047', 'EQ048', 'EQ049', 'EQ051', 'EQ053', 'EQ054', 'EQ055', 'EQ056', 'EQ057', 'EQ058', 'EQ059', 'EQ060', 'EQ061', 'EQ062', 'EQ063', 'EQ064', 'EQ065', 'EQ066', 'EQ067', 'EQ069', 'EQ071', 'EQ072', 'EQ074', 'EQ076', 'EQ077', 'EQ078', 'EQ079', 'EQ080', 'EQ082', 'EQ084', 'EQ085', 'EQ086']
✅ val_final_files  : ['EQ004', 'EQ012', 'EQ014', 'EQ021', 'EQ022', 'EQ027', 'EQ028', 'EQ030', 'EQ037', 'EQ045', 'EQ050', 'EQ068', 'EQ070', 'EQ073', 'EQ075', 'EQ083']
✅ test_final_files : ['EQ007', 'EQ019', 'EQ052', 'EQ081']

✅ Scalers finales listos
XM_FINAL: [[ 1.25773285e-08  1.85184277e-08 -4.83568219e-05  3.44104450e-07
  -1.33830520e-06]]
X_P99ABS_FINAL: [[0.02921616 0.53167486 0.05598981 0.070

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 6
# Dataset cacheado por ventanas
# - Genera ventanas temporales desde cache en RAM
# - TRAIN y VAL usan caches distintos
# - K_per_event fijo evita que eventos largos dominen
# ============================================
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

WIN_LEN = T_BACK

class WindowEventDatasetCached(Dataset):
    def __init__(self, cache, K_per_event=256, fixed=False, seed=123, win_len=T_BACK):
        self.cache = cache
        self.files = list(cache.keys())
        self.K = int(K_per_event)
        self.fixed = bool(fixed)
        self.rng = np.random.RandomState(seed)
        self.win_len = int(win_len)
        self.items = []
        self.refresh()

    def refresh(self):
        self.items = []
        for p in self.files:
            NT = self.cache[p]["NT"]
            if NT < self.win_len + 1:
                continue

            max_t0 = NT - self.win_len
            if self.fixed:
                t0s = self.rng.randint(0, max_t0 + 1, size=self.K)
            else:
                t0s = np.random.randint(0, max_t0 + 1, size=self.K)

            for t0 in t0s:
                self.items.append((p, int(t0)))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, t0 = self.items[idx]
        Xn = self.cache[p]["Xn"]
        Yn = self.cache[p]["Yn"]

        xb = Xn[t0:t0+self.win_len]
        yb = Yn[t0:t0+self.win_len]

        return torch.from_numpy(xb), torch.from_numpy(yb)

def make_window_loader_cached(cache, K_per_event, batch, shuffle, fixed=False, seed=123, num_workers=0):
    ds = WindowEventDatasetCached(
        cache=cache,
        K_per_event=K_per_event,
        fixed=fixed,
        seed=seed,
        win_len=WIN_LEN
    )
    dl = DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=(DEVICE == "cuda")
    )
    return ds, dl

# smoke test
ds_final_smoke, dl_final_smoke = make_window_loader_cached(
    cache_train_final,
    K_per_event=8,
    batch=4,
    shuffle=True,
    fixed=False,
    seed=123,
    num_workers=0
)

xb, yb = next(iter(dl_final_smoke))
print("✅ batch X:", xb.shape, "| batch Y:", yb.shape)

✅ batch X: torch.Size([4, 1024, 5]) | batch Y: torch.Size([4, 1024, 3])


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 7
# Entrenamiento/evaluación GroupKFold del Teacher
# - Fit de scalers SOLO en train de cada fold
# - Val por EVENTO (sin mezcla)
# - Guarda métricas por fold
# ============================================
import os
import time
import copy
import numpy as np
import pandas as pd
import torch

# --------------------------------------------------
# Hiperparámetros CV Teacher
# --------------------------------------------------
CV_TAG = "TEACHER_GKFOLD"
HID = 256
LAY = 2
DROPOUT = 0.0
LR = 1e-3
EPOCHS = 60
BATCH = 32

K_TRAIN_PER_EVENT = 128
K_VAL_PER_EVENT   = 64

lam_scale = 0.005
grad_clip = 1.0
patience  = 10
P_P99 = 99.0

cv_summary_rows = []
cv_histories = {}

def batch_corr_teacher(yhat, ytrue, eps=1e-12):
    yhat = yhat.detach()
    ytrue = ytrue.detach()

    corrs = []
    n_ch = ytrue.shape[-1]

    for ch in range(n_ch):
        t = ytrue[:, :, ch].reshape(-1)
        p = yhat[:, :, ch].reshape(-1)

        t = t - torch.mean(t)
        p = p - torch.mean(p)

        num = torch.sum(t * p)
        den = torch.sqrt(torch.sum(t * t)) * torch.sqrt(torch.sum(p * p)) + eps
        corrs.append(num / den)

    return float(torch.mean(torch.stack(corrs)).item())

for fd in teacher_folds:
    fold_idx = fd["fold"]
    print(f"\n{'='*80}")
    print(f"🚀 TEACHER CV — FOLD {fold_idx}/{len(teacher_folds)}")
    print(f"{'='*80}")

    train_files_fold = fd["train_files"]
    val_files_fold   = fd["val_files"]

    # --------------------------------------------------
    # Scalers SOLO con train del fold
    # --------------------------------------------------
    XM_F, YM_F, X_P99ABS_F, Y_P99ABS_F = fit_global_scalers_mean_p99abs(
        train_files_fold, dt_ref=DT_REF, p=P_P99
    )

    cache_train_fold = build_event_cache(
        train_files_fold, XM_F, YM_F, X_P99ABS_F, Y_P99ABS_F,
        dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
    )

    cache_val_fold = build_event_cache(
        val_files_fold, XM_F, YM_F, X_P99ABS_F, Y_P99ABS_F,
        dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
    )

    ds_tr, dl_tr = make_window_loader_cached(
        cache_train_fold,
        K_per_event=K_TRAIN_PER_EVENT,
        batch=BATCH,
        shuffle=True,
        fixed=False,
        seed=123 + fold_idx,
        num_workers=0
    )

    ds_va, dl_va = make_window_loader_cached(
        cache_val_fold,
        K_per_event=K_VAL_PER_EVENT,
        batch=BATCH,
        shuffle=False,
        fixed=True,
        seed=999 + fold_idx,
        num_workers=0
    )

    print("Ventanas TRAIN:", len(ds_tr))
    print("Ventanas VAL  :", len(ds_va))

    model = TeacherLSTM(
        in_dim=5,
        hid=HID,
        layers=LAY,
        dropout=DROPOUT,
        out_dim=3
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val = float("inf")
    best_state = None
    bad = 0

    fold_history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_corr": [],
        "val_corr": [],
        "lr": [],
        "epoch_time_sec": []
    }

    t_fold_ini = time.time()

    for ep in range(1, EPOCHS + 1):
        t_ep = time.time()

        ds_tr.refresh()

        # ---------------- TRAIN ----------------
        model.train()
        tr_losses, tr_corrs = [], []

        for xb, yb in dl_tr:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
            yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)

            opt.zero_grad(set_to_none=True)

            yhat = model(xb)
            loss_main = torch.mean((yhat - yb) ** 2)
            loss = loss_main + lam_scale * (yhat ** 2).mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            tr_losses.append(float(loss.item()))
            tr_corrs.append(batch_corr_teacher(yhat, yb))

        train_loss = float(np.mean(tr_losses)) if tr_losses else np.nan
        train_corr = float(np.mean(tr_corrs)) if tr_corrs else np.nan

        # ---------------- VAL ----------------
        model.eval()
        va_losses, va_corrs = [], []

        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)

                xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
                yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)

                yhat = model(xb)
                loss_main = torch.mean((yhat - yb) ** 2)
                loss = loss_main + lam_scale * (yhat ** 2).mean()

                va_losses.append(float(loss.item()))
                va_corrs.append(batch_corr_teacher(yhat, yb))

        val_loss = float(np.mean(va_losses)) if va_losses else np.nan
        val_corr = float(np.mean(va_corrs)) if va_corrs else np.nan

        lr_now = float(opt.param_groups[0]["lr"])
        ep_time = float(time.time() - t_ep)

        fold_history["epoch"].append(ep)
        fold_history["train_loss"].append(train_loss)
        fold_history["val_loss"].append(val_loss)
        fold_history["train_corr"].append(train_corr)
        fold_history["val_corr"].append(val_corr)
        fold_history["lr"].append(lr_now)
        fold_history["epoch_time_sec"].append(ep_time)

        print(
            f"[Fold {fold_idx} | {ep:03d}/{EPOCHS}] "
            f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
            f"train_corr={train_corr:.4f} | val_corr={val_corr:.4f} | "
            f"time={ep_time:.1f}s"
        )

        if val_loss < best_val:
            best_val = val_loss
            bad = 0
            best_state = {
                "model": copy.deepcopy(model.state_dict()),
                "epoch": ep,
                "val_loss": val_loss,
                "val_corr": val_corr,
            }
        else:
            bad += 1
            if bad >= patience:
                print(f"⏹️ Early stopping en fold {fold_idx}, época {ep}")
                break

    fold_train_time = float(time.time() - t_fold_ini)

    assert best_state is not None, f"Fold {fold_idx}: no se guardó best_state."
    model.load_state_dict(best_state["model"])

    cv_summary_rows.append({
        "fold": fold_idx,
        "n_train_events": len(fd["train_ids"]),
        "n_val_events": len(fd["val_ids"]),
        "best_epoch": int(best_state["epoch"]),
        "best_val_loss": float(best_state["val_loss"]),
        "best_val_corr": float(best_state["val_corr"]),
        "train_time_sec": fold_train_time,
    })

    cv_histories[f"fold_{fold_idx}"] = pd.DataFrame(fold_history)

    # guardar checkpoint fold
    ckpt_fold_path = os.path.join(RUNS_DIR_B, f"{CV_TAG}_fold{fold_idx}.pt")
    hist_fold_path = os.path.join(RUNS_DIR_B, f"{CV_TAG}_fold{fold_idx}_history.csv")

    torch.save({
        "state_dict": model.state_dict(),
        "cfg": {
            "tag": f"{CV_TAG}_fold{fold_idx}",
            "fold": fold_idx,
            "dt_ref": DT_REF,
            "T_back": T_BACK,
            "hid": HID,
            "layers": LAY,
            "dropout": DROPOUT,
            "lr": LR,
            "batch": BATCH,
            "K_train_per_event": K_TRAIN_PER_EVENT,
            "K_val_per_event": K_VAL_PER_EVENT,
            "best_epoch": int(best_state["epoch"]),
            "best_val_loss": float(best_state["val_loss"]),
            "best_val_corr": float(best_state["val_corr"]),
            "train_ids": fd["train_ids"],
            "val_ids": fd["val_ids"],
            "test_ids_final": TEST_IDS_FINAL,
            "XM": XM_F,
            "YM": YM_F,
            "X_P99ABS": X_P99ABS_F,
            "Y_P99ABS": Y_P99ABS_F,
            "clip_x": CLIP_X,
            "clip_y": CLIP_Y,
        }
    }, ckpt_fold_path)

    pd.DataFrame(fold_history).to_csv(hist_fold_path, index=False)

    print(f"✅ Fold {fold_idx} guardado en:")
    print("   ", ckpt_fold_path)
    print("   ", hist_fold_path)

# --------------------------------------------------
# Resumen final CV
# --------------------------------------------------
cv_summary_df = pd.DataFrame(cv_summary_rows)
cv_summary_path = os.path.join(RUNS_DIR_B, f"{CV_TAG}_summary.csv")
cv_summary_df.to_csv(cv_summary_path, index=False)

print("\n" + "="*80)
print("✅ GROUPKFOLD TEACHER COMPLETADO")
print("="*80)
print(cv_summary_df)
print("\nResumen guardado en:", cv_summary_path)
print("\nPromedios:")
print(cv_summary_df[["best_val_loss", "best_val_corr", "best_epoch"]].mean(numeric_only=True))


🚀 TEACHER CV — FOLD 1/5
Ventanas TRAIN: 8192
Ventanas VAL  : 1024
[Fold 1 | 001/60] train_loss=0.085496 | val_loss=0.265346 | train_corr=0.3512 | val_corr=0.2292 | time=53.4s
[Fold 1 | 002/60] train_loss=0.077974 | val_loss=0.228112 | train_corr=0.3976 | val_corr=0.4444 | time=48.3s
[Fold 1 | 003/60] train_loss=0.068487 | val_loss=0.177071 | train_corr=0.4798 | val_corr=0.5136 | time=75.3s
[Fold 1 | 004/60] train_loss=0.058787 | val_loss=0.187183 | train_corr=0.5786 | val_corr=0.5245 | time=57.6s
[Fold 1 | 005/60] train_loss=0.050544 | val_loss=0.168581 | train_corr=0.6810 | val_corr=0.5527 | time=62.5s
[Fold 1 | 006/60] train_loss=0.039228 | val_loss=0.140843 | train_corr=0.7558 | val_corr=0.7112 | time=76.3s
[Fold 1 | 007/60] train_loss=0.035836 | val_loss=0.130832 | train_corr=0.8022 | val_corr=0.7594 | time=59.9s
[Fold 1 | 008/60] train_loss=0.031619 | val_loss=0.140453 | train_corr=0.8277 | val_corr=0.7543 | time=75.3s
[Fold 1 | 009/60] train_loss=0.027327 | val_loss=0.149697 | t

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 8
# Entrenamiento final oficial del LSTM_Teacher
# - Entrenamiento con TRAIN por evento
# - Validación con VAL por evento
# - Sin mezcla de eventos entre train y val
# - Guarda history real para Bloque 4
# ============================================
import os
import time
import numpy as np
import pandas as pd
import torch

FINAL_TAG = "TEACHER_FINAL_WINDOWS_OFFICIAL"

# --------------------------------------------------
# Hiperparámetros finales
# --------------------------------------------------
HID = 256
LAY = 2
DROPOUT = 0.0
LR = 1e-3
EPOCHS = 80
BATCH = 32

K_TRAIN_PER_EVENT = 128
K_VAL_PER_EVENT   = 64

lam_scale = 0.005
grad_clip = 1.0
patience  = 12

# --------------------------------------------------
# Helper: correlación media por batch
# --------------------------------------------------
def batch_corr_teacher(yhat, ytrue, eps=1e-12):
    """
    Calcula correlación promedio entre predicción y target,
    promediando sobre los 3 canales de salida.
    yhat, ytrue: (B,T,3)
    """
    yhat = yhat.detach()
    ytrue = ytrue.detach()

    corrs = []
    n_ch = ytrue.shape[-1]

    for ch in range(n_ch):
        t = ytrue[:, :, ch].reshape(-1)
        p = yhat[:, :, ch].reshape(-1)

        t = t - torch.mean(t)
        p = p - torch.mean(p)

        num = torch.sum(t * p)
        den = torch.sqrt(torch.sum(t * t)) * torch.sqrt(torch.sum(p * p)) + eps
        corr = num / den
        corrs.append(corr)

    return float(torch.mean(torch.stack(corrs)).item())

# --------------------------------------------------
# Loaders
# --------------------------------------------------
ds_tr_final, dl_tr_final = make_window_loader_cached(
    cache_train_final,
    K_per_event=K_TRAIN_PER_EVENT,
    batch=BATCH,
    shuffle=True,
    fixed=False,
    seed=123,
    num_workers=0
)

ds_va_final, dl_va_final = make_window_loader_cached(
    cache_val_final,
    K_per_event=K_VAL_PER_EVENT,
    batch=BATCH,
    shuffle=False,
    fixed=True,
    seed=999,
    num_workers=0
)

print("✅ Ventanas TRAIN:", len(ds_tr_final))
print("✅ Ventanas VAL  :", len(ds_va_final))

# --------------------------------------------------
# Modelo + optimizador
# --------------------------------------------------
model_final = TeacherLSTM(
    in_dim=5,
    hid=HID,
    layers=LAY,
    dropout=DROPOUT,
    out_dim=3
).to(DEVICE)

opt_final = torch.optim.AdamW(model_final.parameters(), lr=LR)

best_val = float("inf")
best_state = None
bad = 0

history_final = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_corr": [],
    "val_corr": [],
    "lr": [],
    "epoch_time_sec": []
}

t_train_ini = time.time()

for ep in range(1, EPOCHS + 1):
    t_ep = time.time()

    # refresca solo TRAIN para re-samplear ventanas por época
    ds_tr_final.refresh()

    # ---------------- TRAIN ----------------
    model_final.train()
    tr_losses = []
    tr_corrs  = []

    for xb, yb in dl_tr_final:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
        yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)

        opt_final.zero_grad(set_to_none=True)

        yhat = model_final(xb)
        loss_main = torch.mean((yhat - yb) ** 2)
        loss = loss_main + lam_scale * (yhat ** 2).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), grad_clip)
        opt_final.step()

        tr_losses.append(float(loss.item()))
        tr_corrs.append(batch_corr_teacher(yhat, yb))

    train_loss = float(np.mean(tr_losses)) if tr_losses else np.nan
    train_corr = float(np.mean(tr_corrs)) if tr_corrs else np.nan

    # ---------------- VAL ----------------
    model_final.eval()
    va_losses = []
    va_corrs  = []

    with torch.no_grad():
        for xb, yb in dl_va_final:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
            yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)

            yhat = model_final(xb)
            loss_main = torch.mean((yhat - yb) ** 2)
            loss = loss_main + lam_scale * (yhat ** 2).mean()

            va_losses.append(float(loss.item()))
            va_corrs.append(batch_corr_teacher(yhat, yb))

    val_loss = float(np.mean(va_losses)) if va_losses else np.nan
    val_corr = float(np.mean(va_corrs)) if va_corrs else np.nan

    lr_now = float(opt_final.param_groups[0]["lr"])
    ep_time = float(time.time() - t_ep)

    history_final["epoch"].append(ep)
    history_final["train_loss"].append(train_loss)
    history_final["val_loss"].append(val_loss)
    history_final["train_corr"].append(train_corr)
    history_final["val_corr"].append(val_corr)
    history_final["lr"].append(lr_now)
    history_final["epoch_time_sec"].append(ep_time)

    print(
        f"[{ep:03d}/{EPOCHS}] "
        f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
        f"train_corr={train_corr:.4f} | val_corr={val_corr:.4f} | "
        f"time={ep_time:.1f}s"
    )

    # early stopping por VAL
    if val_loss < best_val:
        best_val = val_loss
        bad = 0
        best_state = {
            "model": {k: v.detach().cpu().clone() for k, v in model_final.state_dict().items()},
            "epoch": ep,
            "val_loss": val_loss
        }
    else:
        bad += 1
        if bad >= patience:
            print(f"⏹️ Early stopping en época {ep} (patience={patience})")
            break

total_train_time = float(time.time() - t_train_ini)

# --------------------------------------------------
# Restaurar mejor modelo
# --------------------------------------------------
assert best_state is not None, "No se guardó best_state."
model_final.load_state_dict(best_state["model"])

# --------------------------------------------------
# Guardado
# --------------------------------------------------
ckpt_path_final = os.path.join(RUNS_DIR_B, f"{FINAL_TAG}.pt")
hist_path_final = os.path.join(RUNS_DIR_B, f"{FINAL_TAG}_history.csv")

torch.save({
    "state_dict": model_final.state_dict(),
    "cfg": {
        "tag": FINAL_TAG,
        "dt_ref": DT_REF,
        "T_back": T_BACK,
        "hid": HID,
        "layers": LAY,
        "dropout": DROPOUT,
        "lr": LR,
        "batch": BATCH,
        "K_train_per_event": K_TRAIN_PER_EVENT,
        "K_val_per_event": K_VAL_PER_EVENT,
        "best_epoch": int(best_state["epoch"]),
        "best_val_loss": float(best_state["val_loss"]),
        "train_ids_final": TRAIN_IDS_FINAL,
        "val_ids_final": VAL_IDS_FINAL,
        "test_ids_final": TEST_IDS_FINAL,
        "XM": XM_FINAL,
        "YM": YM_FINAL,
        "X_P99ABS": X_P99ABS_FINAL,
        "Y_P99ABS": Y_P99ABS_FINAL,
        "clip_x": CLIP_X,
        "clip_y": CLIP_Y,
        "total_train_time_sec": total_train_time,
    }
}, ckpt_path_final)

pd.DataFrame(history_final).to_csv(hist_path_final, index=False)

print("\n✅ Entrenamiento final Teacher terminado.")
print("Best epoch:", best_state["epoch"])
print("Best val_loss:", best_state["val_loss"])
print("Checkpoint:", ckpt_path_final)
print("History:", hist_path_final)

✅ Ventanas TRAIN: 8192
✅ Ventanas VAL  : 1024
[001/80] train_loss=0.098057 | val_loss=0.121695 | train_corr=0.3411 | val_corr=0.5223 | time=56.9s
[002/80] train_loss=0.083259 | val_loss=0.118454 | train_corr=0.4544 | val_corr=0.4299 | time=64.4s
[003/80] train_loss=0.072175 | val_loss=0.084876 | train_corr=0.5385 | val_corr=0.6624 | time=47.3s
[004/80] train_loss=0.063354 | val_loss=0.079605 | train_corr=0.6816 | val_corr=0.6667 | time=53.2s
[005/80] train_loss=0.049749 | val_loss=0.087546 | train_corr=0.7179 | val_corr=0.7149 | time=50.3s
[006/80] train_loss=0.042147 | val_loss=0.074209 | train_corr=0.7883 | val_corr=0.7332 | time=53.6s
[007/80] train_loss=0.035603 | val_loss=0.071303 | train_corr=0.8101 | val_corr=0.7074 | time=53.5s
[008/80] train_loss=0.036468 | val_loss=0.074921 | train_corr=0.8102 | val_corr=0.7652 | time=73.7s
[009/80] train_loss=0.025812 | val_loss=0.065387 | train_corr=0.8591 | val_corr=0.7777 | time=73.9s
[010/80] train_loss=0.026923 | val_loss=0.063218 | tra

## 4.- Evaluacion final de Teacher

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 9
# Evaluación final del Teacher en los 4 eventos holdout
# - predicción full-event mediante overlap-add
# - cálculo de métricas físicas
# - exportación CSV
# ============================================
import numpy as np
import pandas as pd
import torch
import time
import os

def active_strict_indices(y_true, dt, warmup_s=1.0, tail_margin_s=0.5):
    a = np.asarray(y_true[:, 0], dtype=np.float64)
    peak = np.max(np.abs(a)) + 1e-12

    thr_on = 0.05 * peak
    idx_on = np.where(np.abs(a) >= thr_on)[0]
    onset = int(idx_on[0]) if len(idx_on) else 0

    thr_tail = 0.02 * peak
    idx_tail = np.where(np.abs(a) >= thr_tail)[0]
    last = int(idx_tail[-1]) if len(idx_tail) else (len(a) - 1)

    warm  = int(round(warmup_s / float(dt)))
    tailm = int(round(tail_margin_s / float(dt)))

    i0 = min(len(a) - 1, max(0, onset + warm))
    i1 = max(i0 + 1, min(len(a), last - tailm + 1))
    return i0, i1

def regression_metrics_block(y_true, y_pred, i0, i1):
    yt = np.asarray(y_true[i0:i1], dtype=np.float64)
    yp = np.asarray(y_pred[i0:i1], dtype=np.float64)

    rows = []
    for ch, name in enumerate(Y_NAMES):
        t = yt[:, ch]
        p = yp[:, ch]

        rmse = float(np.sqrt(np.mean((p - t) ** 2)))
        mae  = float(np.mean(np.abs(p - t)))
        bias = float(np.mean(p - t))
        corr = float(np.corrcoef(t, p)[0, 1]) if (np.std(t) > 0 and np.std(p) > 0) else np.nan

        peak_t = float(np.max(np.abs(t)))
        peak_p = float(np.max(np.abs(p)))
        peak_err_rel = float(np.abs(peak_p - peak_t) / (peak_t + 1e-12))

        rows.append({
            "target": name,
            "RMSE": rmse,
            "MAE": mae,
            "Bias": bias,
            "Corr": corr,
            "PeakTrueAbs": peak_t,
            "PeakPredAbs": peak_p,
            "PeakErrRel": peak_err_rel
        })
    return rows

@torch.no_grad()
def predict_window_model_from_cache(model, cache_item, stride_eval=16):
    rid = cache_item["rid"]
    Xn = cache_item["Xn"]
    Yr = cache_item["Yr"]

    NT = Xn.shape[0]
    ypred = np.zeros((NT, 3), dtype=np.float32)
    cnt   = np.zeros((NT, 1), dtype=np.float32)

    for t0 in range(0, NT - T_BACK + 1, stride_eval):
        xb = torch.from_numpy(Xn[t0:t0+T_BACK][None, :, :]).to(DEVICE)
        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)

        yh = model(xb).detach().cpu().numpy()[0]
        ypred[t0:t0+T_BACK] += yh
        cnt[t0:t0+T_BACK] += 1.0

    # asegura cobertura del final
    if cnt[-1, 0] == 0 and NT >= T_BACK:
        t0 = NT - T_BACK
        xb = torch.from_numpy(Xn[t0:t0+T_BACK][None, :, :]).to(DEVICE)
        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)

        yh = model(xb).detach().cpu().numpy()[0]
        ypred[t0:t0+T_BACK] += yh
        cnt[t0:t0+T_BACK] += 1.0

    ypred /= np.maximum(cnt, 1.0)

    Yp = (ypred * (Y_P99ABS_FINAL + EPS)) + YM_FINAL
    Yp = np.nan_to_num(Yp.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    return rid, Yr.astype(np.float32), Yp.astype(np.float32), DT_REF

# --------------------------------------------------
# Evaluación final
# --------------------------------------------------
STRIDE_EVAL_FINAL = 16
ROWS_FINAL = []
RESULTS_FINAL = {}

for p in test_final_files:
    t0_eval = time.perf_counter()

    rid, Yr, Yp, dt2 = predict_window_model_from_cache(
        model_final,
        cache_test_final[p],
        stride_eval=STRIDE_EVAL_FINAL
    )

    eval_time = time.perf_counter() - t0_eval

    i0, i1 = active_strict_indices(Yr, dt2, warmup_s=1.0, tail_margin_s=0.5)
    mets = regression_metrics_block(Yr, Yp, i0, i1)

    RESULTS_FINAL[rid] = {
        "Yr": Yr,
        "Yp": Yp,
        "dt_ref": dt2,
        "i0": i0,
        "i1": i1,
        "mets": mets
    }

    print(f"\n=== FINAL TEST | {rid} | ACTIVE [{i0}:{i1}] ===")
    for row in mets:
        print(
            f"  {row['target']:8s} | RMSE={row['RMSE']:.6g} | "
            f"MAE={row['MAE']:.6g} | Bias={row['Bias']:+.3g} | "
            f"Corr={row['Corr']:+.3f} | PeakErrRel={row['PeakErrRel']:.3f}"
        )

        ROWS_FINAL.append({
            "Model": FINAL_TAG,
            "Event": rid,
            "dt_ref": dt2,
            "active_len": int(i1 - i0),
            "TrainTime_s": time_final,
            "EvalTime_s": eval_time,
            **row
        })

df_final_metrics = pd.DataFrame(ROWS_FINAL)

print("\n=== PROMEDIO POR TARGET ===")
display(df_final_metrics.groupby("target")[["RMSE","MAE","Bias","Corr","PeakErrRel","TrainTime_s","EvalTime_s"]].mean())

CSV_FINAL = os.path.join(RUNS_DIR_B, f"{FINAL_TAG}_FINAL_METRICS.csv")
df_final_metrics.to_csv(CSV_FINAL, index=False)

print("\n✅ CSV guardado:", CSV_FINAL)


=== FINAL TEST | EQ007 | ACTIVE [385:3197] ===
  a_base_x | RMSE=0.262106 | MAE=0.144846 | Bias=+0.0258 | Corr=+0.917 | PeakErrRel=0.455
  Fx       | RMSE=29.5987 | MAE=16.5568 | Bias=+8.4 | Corr=+0.955 | PeakErrRel=0.290
  Mz       | RMSE=503.046 | MAE=343.318 | Bias=-321 | Corr=+0.720 | PeakErrRel=0.454

=== FINAL TEST | EQ019 | ACTIVE [1778:11843] ===
  a_base_x | RMSE=0.859186 | MAE=0.442799 | Bias=+0.014 | Corr=+0.920 | PeakErrRel=0.197
  Fx       | RMSE=105.777 | MAE=70.396 | Bias=+19.5 | Corr=+0.952 | PeakErrRel=0.219
  Mz       | RMSE=3421.69 | MAE=2627.05 | Bias=-673 | Corr=+0.809 | PeakErrRel=0.233

=== FINAL TEST | EQ052 | ACTIVE [4248:12100] ===
  a_base_x | RMSE=0.107132 | MAE=0.0859293 | Bias=+0.0201 | Corr=+0.871 | PeakErrRel=0.329
  Fx       | RMSE=22.228 | MAE=18.2767 | Bias=+6.38 | Corr=+0.847 | PeakErrRel=0.056
  Mz       | RMSE=917.358 | MAE=749.774 | Bias=-245 | Corr=+0.859 | PeakErrRel=0.310

=== FINAL TEST | EQ081 | ACTIVE [3216:14232] ===
  a_base_x | RMSE=0.21

,RMSE,MAE,Bias,Corr,PeakErrRel,TrainTime_s,EvalTime_s
target,,,,,,,
Fx,46.848085,31.323980,10.428847,0.918982,0.151491,3962.422122,62.868946
Mz,1462.649149,1130.125375,-383.576254,0.785386,0.400415,3962.422122,62.868946
a_base_x,0.361150,0.195104,0.021078,0.906839,0.361725,3962.422122,62.868946



✅ CSV guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/TEACHER_FINAL_WINDOWS_OFFICIAL_FINAL_METRICS.csv


# 5.- Visualización Final

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 10
# Plots finales del Teacher
# - gráficos true vs pred
# - zoom en ACTIVE_STRICT
# - vista global opcional
# - guarda todo en un PDF
# ============================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os

PLOT_GLOBAL = True
PLOT_ACTIVE_ONLY = True
FIGSIZE = (12, 4)
DPI = 180

def _get_metrics_dict(out):
    md = {}
    for row in out["mets"]:
        md[row["target"]] = row
    return md

def _plot_one(pdf, t, y_true, y_pred, title, ylab):
    fig = plt.figure(figsize=FIGSIZE)
    plt.plot(t, y_true, label="true")
    plt.plot(t, y_pred, label="pred")
    plt.grid(True, alpha=0.3)
    plt.xlabel("t [s]")
    plt.ylabel(ylab)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    pdf.savefig(fig, dpi=DPI)
    plt.close(fig)

PDF_FINAL = os.path.join(RUNS_DIR_B, f"{FINAL_TAG}_FINAL_PLOTS.pdf")

with PdfPages(PDF_FINAL) as pdf:
    # portada
    fig = plt.figure(figsize=(12, 3))
    plt.axis("off")
    plt.text(
        0.01, 0.7,
        f"{FINAL_TAG}\n"
        f"source=RESULTS_FINAL\n"
        f"PLOT_ACTIVE_ONLY={PLOT_ACTIVE_ONLY} | PLOT_GLOBAL={PLOT_GLOBAL}\n"
        f"events={list(RESULTS_FINAL.keys())}",
        fontsize=12
    )
    plt.tight_layout()
    pdf.savefig(fig, dpi=DPI)
    plt.close(fig)

    for rid, out in RESULTS_FINAL.items():
        Yr = out["Yr"]
        Yp = out["Yp"]
        i0, i1 = int(out["i0"]), int(out["i1"])
        dt2 = float(out["dt_ref"])

        t_all = np.arange(Yr.shape[0]) * dt2
        sl = slice(i0, i1)

        md = _get_metrics_dict(out)

        # ---------------- ACTIVE ----------------
        if PLOT_ACTIVE_ONLY:
            _plot_one(
                pdf,
                t_all[sl], Yr[sl, 0], Yp[sl, 0],
                title=f"{rid} | a_base_x | ACTIVE [{i0}:{i1}] | RMSE={md['a_base_x']['RMSE']:.4g} | Corr={md['a_base_x']['Corr']:+.3f}",
                ylab="a_base_x"
            )

            _plot_one(
                pdf,
                t_all[sl], Yr[sl, 1], Yp[sl, 1],
                title=f"{rid} | Fx | ACTIVE [{i0}:{i1}] | RMSE={md['Fx']['RMSE']:.4g} | Corr={md['Fx']['Corr']:+.3f}",
                ylab="Fx"
            )

            _plot_one(
                pdf,
                t_all[sl], Yr[sl, 2], Yp[sl, 2],
                title=f"{rid} | Mz | ACTIVE [{i0}:{i1}] | RMSE={md['Mz']['RMSE']:.4g} | Corr={md['Mz']['Corr']:+.3f}",
                ylab="Mz"
            )

        # ---------------- GLOBAL ----------------
        if PLOT_GLOBAL:
            _plot_one(
                pdf,
                t_all, Yr[:, 0], Yp[:, 0],
                title=f"{rid} | a_base_x | GLOBAL",
                ylab="a_base_x"
            )

            _plot_one(
                pdf,
                t_all, Yr[:, 1], Yp[:, 1],
                title=f"{rid} | Fx | GLOBAL",
                ylab="Fx"
            )

            _plot_one(
                pdf,
                t_all, Yr[:, 2], Yp[:, 2],
                title=f"{rid} | Mz | GLOBAL",
                ylab="Mz"
            )

print("✅ PDF guardado:", PDF_FINAL)

✅ PDF guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/TEACHER_FINAL_WINDOWS_OFFICIAL_FINAL_PLOTS.pdf


# 6.- Comparación metodológica A/B/C

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 9
# Split 50/10/4 para comparación metodológica
# - 50 eventos para train
# - 10 eventos para validación
# - 4 eventos para test comparativo
# ============================================
import os
import random

SEED_SPLIT_COMPARE = 123

def record_id_from_path(p):
    return os.path.splitext(os.path.basename(p))[0]

all_ids_compare = sorted([record_id_from_path(p) for p in ALL_FILES])
assert len(all_ids_compare) >= 64, f"Se esperaban al menos 64 eventos, encontrados: {len(all_ids_compare)}"

random.seed(SEED_SPLIT_COMPARE)
ids_shuffled = all_ids_compare.copy()
random.shuffle(ids_shuffled)

TRAIN_IDS_CMP = sorted(ids_shuffled[:50])
VAL_IDS_CMP   = sorted(ids_shuffled[50:60])
TEST_IDS_CMP  = sorted(ids_shuffled[60:64])

train_files_cmp = [p for p in ALL_FILES if record_id_from_path(p) in TRAIN_IDS_CMP]
val_files_cmp   = [p for p in ALL_FILES if record_id_from_path(p) in VAL_IDS_CMP]
test_files_cmp  = [p for p in ALL_FILES if record_id_from_path(p) in TEST_IDS_CMP]

print("=== SPLIT COMPARATIVO 50/10/4 ===")
print("TRAIN (50):", TRAIN_IDS_CMP)
print("VAL   (10):", VAL_IDS_CMP)
print("TEST  (4): ", TEST_IDS_CMP)

assert len(train_files_cmp) == 50
assert len(val_files_cmp) == 10
assert len(test_files_cmp) == 4

print("\n✅ Split comparativo listo.")

=== SPLIT COMPARATIVO 50/10/4 ===
TRAIN (50): ['EQ001', 'EQ002', 'EQ004', 'EQ005', 'EQ008', 'EQ009', 'EQ010', 'EQ011', 'EQ012', 'EQ013', 'EQ014', 'EQ015', 'EQ016', 'EQ018', 'EQ021', 'EQ022', 'EQ023', 'EQ025', 'EQ026', 'EQ028', 'EQ030', 'EQ031', 'EQ032', 'EQ033', 'EQ034', 'EQ035', 'EQ036', 'EQ039', 'EQ040', 'EQ041', 'EQ042', 'EQ043', 'EQ044', 'EQ045', 'EQ046', 'EQ047', 'EQ048', 'EQ049', 'EQ050', 'EQ051', 'EQ053', 'EQ054', 'EQ055', 'EQ057', 'EQ059', 'EQ060', 'EQ061', 'EQ062', 'EQ063', 'EQ064']
VAL   (10): ['EQ003', 'EQ024', 'EQ027', 'EQ029', 'EQ037', 'EQ038', 'EQ056', 'EQ058', 'EQ065', 'EQ066']
TEST  (4):  ['EQ006', 'EQ007', 'EQ019', 'EQ052']

✅ Split comparativo listo.


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 10
# Scalers y caché para comparación A/B/C
# - Fit SOLO en TRAIN(50)
# - Construcción de caché para train/val/test
# ============================================
import numpy as np

P_P99_CMP = 99.0

XM_CMP, YM_CMP, X_P99ABS_CMP, Y_P99ABS_CMP = fit_global_scalers_mean_p99abs(
    train_files_cmp,
    dt_ref=DT_REF,
    p=P_P99_CMP
)

print("✅ Scalers comparación listos")
print("XM_CMP:", XM_CMP)
print("X_P99ABS_CMP:", X_P99ABS_CMP)
print("YM_CMP:", YM_CMP)
print("Y_P99ABS_CMP:", Y_P99ABS_CMP)

cache_tr_cmp = build_event_cache(
    train_files_cmp, XM_CMP, YM_CMP, X_P99ABS_CMP, Y_P99ABS_CMP,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

cache_va_cmp = build_event_cache(
    val_files_cmp, XM_CMP, YM_CMP, X_P99ABS_CMP, Y_P99ABS_CMP,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

cache_te_cmp = build_event_cache(
    test_files_cmp, XM_CMP, YM_CMP, X_P99ABS_CMP, Y_P99ABS_CMP,
    dt_ref=DT_REF, clip_x=CLIP_X, clip_y=CLIP_Y
)

print("\n✅ cache_tr_cmp:", len(cache_tr_cmp), "eventos")
print("✅ cache_va_cmp:", len(cache_va_cmp), "eventos")
print("✅ cache_te_cmp:", len(cache_te_cmp), "eventos")

✅ Scalers comparación listos
XM_CMP: [[ 3.0767939e-08  8.0930391e-08  1.3753152e-04  1.0914406e-07
  -1.2091641e-06]]
X_P99ABS_CMP: [[0.04064454 0.6788166  0.08297294 0.10394356 1.1683426 ]]
YM_CMP: [[ 1.4164635e-04  3.1870827e-02 -1.4484229e+00]]
Y_P99ABS_CMP: [[3.1863430e+00 4.9351486e+02 1.4135125e+04]]

✅ cache_tr_cmp: 50 eventos
✅ cache_va_cmp: 10 eventos
✅ cache_te_cmp: 4 eventos


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 11
# Datasets para comparación A/B/C
# - ventanas cacheadas
# - secuencias completas cacheadas
# ============================================
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# Dataset full-event cacheado
# --------------------------------------------------
class FullEventDatasetCached(Dataset):
    def __init__(self, cache):
        self.cache = cache
        self.files = list(cache.keys())

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        p = self.files[idx]
        item = self.cache[p]
        return {
            "rid": item["rid"],
            "X": torch.from_numpy(item["Xn"]),   # (T,5)
            "Y": torch.from_numpy(item["Yn"]),   # (T,3)
            "NT": item["NT"],
            "path": p,
        }

def full_event_collate(batch):
    assert len(batch) == 1, "Para eventos completos usa batch_size=1"
    return batch[0]

# --------------------------------------------------
# Loaders ventanas
# --------------------------------------------------
K_TRAIN_PER_EVENT_CMP = 128
K_VAL_PER_EVENT_CMP   = 64
BATCH_CMP = 32

ds_w_tr_cmp, dl_w_tr_cmp = make_window_loader_cached(
    cache_tr_cmp,
    K_per_event=K_TRAIN_PER_EVENT_CMP,
    batch=BATCH_CMP,
    shuffle=True,
    fixed=False,
    seed=123,
    num_workers=0
)

ds_w_va_cmp, dl_w_va_cmp = make_window_loader_cached(
    cache_va_cmp,
    K_per_event=K_VAL_PER_EVENT_CMP,
    batch=BATCH_CMP,
    shuffle=False,
    fixed=True,
    seed=999,
    num_workers=0
)

# --------------------------------------------------
# Loaders full-event
# --------------------------------------------------
ds_f_tr_cmp = FullEventDatasetCached(cache_tr_cmp)
dl_f_tr_cmp = DataLoader(
    ds_f_tr_cmp,
    batch_size=1,
    shuffle=True,
    collate_fn=full_event_collate,
    num_workers=0
)

ds_f_va_cmp = FullEventDatasetCached(cache_va_cmp)
dl_f_va_cmp = DataLoader(
    ds_f_va_cmp,
    batch_size=1,
    shuffle=False,
    collate_fn=full_event_collate,
    num_workers=0
)

# smoke tests
xb_cmp, yb_cmp = next(iter(dl_w_tr_cmp))
print("✅ Window batch:", xb_cmp.shape, yb_cmp.shape)

sample_full_cmp = next(iter(dl_f_tr_cmp))
print("✅ Full event sample:", sample_full_cmp["rid"], sample_full_cmp["X"].shape, sample_full_cmp["Y"].shape)

✅ Window batch: torch.Size([32, 1024, 5]) torch.Size([32, 1024, 3])
✅ Full event sample: EQ030 torch.Size([18000, 5]) torch.Size([18000, 3])


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 12
# Helpers de evaluación comparativa A/B/C
# - métricas
# - predicción por ventanas
# - predicción por secuencia completa
# ============================================
import numpy as np
import torch
import pandas as pd
import time

def active_strict_indices_cmp(y_true, dt, warmup_s=1.0, tail_margin_s=0.5):
    a = np.asarray(y_true[:, 0], dtype=np.float64)
    peak = np.max(np.abs(a)) + 1e-12

    thr_on = 0.05 * peak
    idx_on = np.where(np.abs(a) >= thr_on)[0]
    onset = int(idx_on[0]) if len(idx_on) else 0

    thr_tail = 0.02 * peak
    idx_tail = np.where(np.abs(a) >= thr_tail)[0]
    last = int(idx_tail[-1]) if len(idx_tail) else (len(a) - 1)

    warm  = int(round(warmup_s / float(dt)))
    tailm = int(round(tail_margin_s / float(dt)))

    i0 = min(len(a) - 1, max(0, onset + warm))
    i1 = max(i0 + 1, min(len(a), last - tailm + 1))
    return i0, i1

def regression_metrics_block_cmp(y_true, y_pred, i0, i1):
    yt = np.asarray(y_true[i0:i1], dtype=np.float64)
    yp = np.asarray(y_pred[i0:i1], dtype=np.float64)

    rows = []
    for ch, name in enumerate(Y_NAMES):
        t = yt[:, ch]
        p = yp[:, ch]

        rmse = float(np.sqrt(np.mean((p - t) ** 2)))
        mae  = float(np.mean(np.abs(p - t)))
        bias = float(np.mean(p - t))
        corr = float(np.corrcoef(t, p)[0, 1]) if (np.std(t) > 0 and np.std(p) > 0) else np.nan

        peak_t = float(np.max(np.abs(t)))
        peak_p = float(np.max(np.abs(p)))
        peak_err_rel = float(np.abs(peak_p - peak_t) / (peak_t + 1e-12))

        rows.append({
            "target": name,
            "RMSE": rmse,
            "MAE": mae,
            "Bias": bias,
            "Corr": corr,
            "PeakTrueAbs": peak_t,
            "PeakPredAbs": peak_p,
            "PeakErrRel": peak_err_rel
        })
    return rows

@torch.no_grad()
def predict_window_model_from_cache_cmp(model, cache_item, stride_eval=16):
    rid = cache_item["rid"]
    Xn = cache_item["Xn"]
    Yr = cache_item["Yr"]

    NT = Xn.shape[0]
    ypred = np.zeros((NT, 3), dtype=np.float32)
    cnt   = np.zeros((NT, 1), dtype=np.float32)

    for t0 in range(0, NT - T_BACK + 1, stride_eval):
        xb = torch.from_numpy(Xn[t0:t0+T_BACK][None, :, :]).to(DEVICE)
        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
        yh = model(xb).detach().cpu().numpy()[0]
        ypred[t0:t0+T_BACK] += yh
        cnt[t0:t0+T_BACK] += 1.0

    if cnt[-1, 0] == 0 and NT >= T_BACK:
        t0 = NT - T_BACK
        xb = torch.from_numpy(Xn[t0:t0+T_BACK][None, :, :]).to(DEVICE)
        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
        yh = model(xb).detach().cpu().numpy()[0]
        ypred[t0:t0+T_BACK] += yh
        cnt[t0:t0+T_BACK] += 1.0

    ypred /= np.maximum(cnt, 1.0)

    Yp = (ypred * (Y_P99ABS_CMP + EPS)) + YM_CMP
    Yp = np.nan_to_num(Yp.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    return rid, Yr.astype(np.float32), Yp.astype(np.float32), DT_REF

@torch.no_grad()
def predict_fullseq_model_from_cache_cmp(model, cache_item, tbptt_chunk=1024):
    rid = cache_item["rid"]
    Xn = cache_item["Xn"]
    Yr = cache_item["Yr"]

    xb = torch.from_numpy(Xn).to(DEVICE)
    model.eval()

    preds = []
    h = None
    T = xb.shape[0]

    for s in range(0, T, tbptt_chunk):
        e = min(T, s + tbptt_chunk)
        x_chunk = xb[s:e].unsqueeze(0)
        y_chunk, h = model.lstm(x_chunk, h)
        y_chunk = model.head(y_chunk)
        preds.append(y_chunk.squeeze(0).detach().cpu())

        if h is not None:
            h = tuple(v.detach() for v in h)

    Yp_n = torch.cat(preds, dim=0).numpy()
    Yp = (Yp_n * (Y_P99ABS_CMP + EPS)) + YM_CMP
    Yp = np.nan_to_num(Yp.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    return rid, Yr.astype(np.float32), Yp.astype(np.float32), DT_REF

In [ ]:
# ============================================
# NOTEBOOK B — CELDA 13
# Modelo A: entrenamiento por ventanas
# ============================================
import os
import time
import numpy as np
import torch

TAG_A = "TEACHER_A_WINDOWS"

HID_CMP = 256
LAY_CMP = 2
DROPOUT_CMP = 0.0
LR_CMP = 1e-3
EPOCHS_CMP = 80
lam_scale_cmp = 0.005
grad_clip_cmp = 1.0
patience_cmp = 12

model_A = TeacherLSTM(
    in_dim=5,
    hid=HID_CMP,
    layers=LAY_CMP,
    dropout=DROPOUT_CMP,
    out_dim=3
).to(DEVICE)

opt_A = torch.optim.AdamW(model_A.parameters(), lr=LR_CMP)

best_val_A = float("inf")
best_state_A = None
bad = 0
t0_train = time.perf_counter()

for ep in range(1, EPOCHS_CMP + 1):
    ds_w_tr_cmp.refresh()

    model_A.train()
    tr_losses = []

    for xb, yb in dl_w_tr_cmp:
        xb = torch.nan_to_num(xb.to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
        yb = torch.nan_to_num(yb.to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)

        pred = model_A(xb)
        loss = mse_loss(pred, yb) + lam_scale_cmp * scale_loss_global(pred, yb)

        if not torch.isfinite(loss):
            continue

        opt_A.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_A.parameters(), grad_clip_cmp)
        opt_A.step()

        tr_losses.append(float(loss.detach().cpu().item()))

    trL = float(np.mean(tr_losses)) if tr_losses else float("inf")

    model_A.eval()
    va_losses = []
    with torch.no_grad():
        for xb, yb in dl_w_va_cmp:
            xb = torch.nan_to_num(xb.to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
            yb = torch.nan_to_num(yb.to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)

            pred = model_A(xb)
            loss = mse_loss(pred, yb) + lam_scale_cmp * scale_loss_global(pred, yb)

            if torch.isfinite(loss):
                va_losses.append(float(loss.detach().cpu().item()))

    vaL = float(np.mean(va_losses)) if va_losses else float("inf")

    if vaL < best_val_A - 1e-6:
        best_val_A = vaL
        best_state_A = {k: v.detach().cpu().clone() for k, v in model_A.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if ep % 5 == 0 or ep == 1:
        print(f"[{TAG_A}] ep{ep:03d} train={trL:.4f} val={vaL:.4f} best={best_val_A:.4f} bad={bad}")

    if bad >= patience_cmp:
        print("Early stop.")
        break

if best_state_A is not None:
    model_A.load_state_dict(best_state_A)

time_A = time.perf_counter() - t0_train

PT_A = os.path.join(RUNS_DIR_B, f"{TAG_A}.pt")
torch.save({
    "state_dict": model_A.state_dict(),
    "cfg": {
        "tag": TAG_A,
        "hid": HID_CMP,
        "layers": LAY_CMP,
        "dropout": DROPOUT_CMP,
        "lr": LR_CMP,
        "epochs_ran": ep,
        "train_mode": "windows",
        "best_val": best_val_A,
        "XM": XM_CMP,
        "YM": YM_CMP,
        "X_P99ABS": X_P99ABS_CMP,
        "Y_P99ABS": Y_P99ABS_CMP
    }
}, PT_A)

print(f"✅ {TAG_A} guardado:", PT_A)
print(f"⏱️ Tiempo entrenamiento A: {time_A:.1f} s")

[TEACHER_A_WINDOWS] ep001 train=0.0760 val=0.1238 best=0.1238 bad=0
[TEACHER_A_WINDOWS] ep005 train=0.0420 val=0.0976 best=0.0976 bad=0
[TEACHER_A_WINDOWS] ep010 train=0.0229 val=0.0566 best=0.0566 bad=0
[TEACHER_A_WINDOWS] ep015 train=0.0162 val=0.0512 best=0.0452 bad=2
[TEACHER_A_WINDOWS] ep020 train=0.0114 val=0.0411 best=0.0411 bad=0
[TEACHER_A_WINDOWS] ep025 train=0.0076 val=0.0446 best=0.0408 bad=3
[TEACHER_A_WINDOWS] ep030 train=0.0087 val=0.0475 best=0.0408 bad=8
Early stop.
✅ TEACHER_A_WINDOWS guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/TEACHER_A_WINDOWS.pt
⏱️ Tiempo entrenamiento A: 884.5 s


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 14
# Modelo B: entrenamiento por secuencias completas (TBPTT)
# ============================================
import os
import time
import numpy as np
import torch

TAG_B = "TEACHER_B_FULLSEQ"

LR_B = 1e-3
EPOCHS_B = 80
TBPTT_CHUNK = 1024
lam_scale_B = 0.005
grad_clip_B = 1.0
patience_B = 12

def train_one_full_event(model, opt, X, Y, tbptt_chunk=1024, train=True):
    X = torch.nan_to_num(X.to(DEVICE), nan=0.0, posinf=0.0, neginf=0.0)
    Y = torch.nan_to_num(Y.to(DEVICE), nan=0.0, posinf=0.0, neginf=0.0)

    T = X.shape[0]
    h = None
    losses = []

    if train:
        model.train()
    else:
        model.eval()

    for s in range(0, T, tbptt_chunk):
        e = min(T, s + tbptt_chunk)
        xb = X[s:e].unsqueeze(0)
        yb = Y[s:e].unsqueeze(0)

        if train:
            y_seq, h = model.lstm(xb, h)
            pred = model.head(y_seq)
            loss = mse_loss(pred, yb) + lam_scale_B * scale_loss_global(pred, yb)

            if not torch.isfinite(loss):
                if h is not None:
                    h = tuple(v.detach() for v in h)
                continue

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_B)
            opt.step()
        else:
            with torch.no_grad():
                y_seq, h = model.lstm(xb, h)
                pred = model.head(y_seq)
                loss = mse_loss(pred, yb) + lam_scale_B * scale_loss_global(pred, yb)

        losses.append(float(loss.detach().cpu().item()))

        if h is not None:
            h = tuple(v.detach() for v in h)

    return float(np.mean(losses)) if losses else float("inf")

model_B = TeacherLSTM(
    in_dim=5,
    hid=HID_CMP,
    layers=LAY_CMP,
    dropout=DROPOUT_CMP,
    out_dim=3
).to(DEVICE)

opt_B = torch.optim.AdamW(model_B.parameters(), lr=LR_B)

best_val_B = float("inf")
best_state_B = None
bad = 0
t0_train = time.perf_counter()

for ep in range(1, EPOCHS_B + 1):
    tr_losses = []
    for batch in dl_f_tr_cmp:
        tr_losses.append(
            train_one_full_event(
                model_B, opt_B,
                batch["X"], batch["Y"],
                tbptt_chunk=TBPTT_CHUNK,
                train=True
            )
        )

    trL = float(np.mean(tr_losses)) if tr_losses else float("inf")

    va_losses = []
    for batch in dl_f_va_cmp:
        va_losses.append(
            train_one_full_event(
                model_B, opt_B,
                batch["X"], batch["Y"],
                tbptt_chunk=TBPTT_CHUNK,
                train=False
            )
        )

    vaL = float(np.mean(va_losses)) if va_losses else float("inf")

    if vaL < best_val_B - 1e-6:
        best_val_B = vaL
        best_state_B = {k: v.detach().cpu().clone() for k, v in model_B.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if ep % 5 == 0 or ep == 1:
        print(f"[{TAG_B}] ep{ep:03d} train={trL:.4f} val={vaL:.4f} best={best_val_B:.4f} bad={bad}")

    if bad >= patience_B:
        print("Early stop.")
        break

if best_state_B is not None:
    model_B.load_state_dict(best_state_B)

time_B = time.perf_counter() - t0_train

PT_B = os.path.join(RUNS_DIR_B, f"{TAG_B}.pt")
torch.save({
    "state_dict": model_B.state_dict(),
    "cfg": {
        "tag": TAG_B,
        "hid": HID_CMP,
        "layers": LAY_CMP,
        "dropout": DROPOUT_CMP,
        "lr": LR_B,
        "epochs_ran": ep,
        "train_mode": "fullseq_tbptt",
        "tbptt_chunk": TBPTT_CHUNK,
        "best_val": best_val_B,
        "XM": XM_CMP,
        "YM": YM_CMP,
        "X_P99ABS": X_P99ABS_CMP,
        "Y_P99ABS": Y_P99ABS_CMP
    }
}, PT_B)

print(f"✅ {TAG_B} guardado:", PT_B)
print(f"⏱️ Tiempo entrenamiento B: {time_B:.1f} s")

[TEACHER_B_FULLSEQ] ep001 train=0.1008 val=0.1218 best=0.1218 bad=0
[TEACHER_B_FULLSEQ] ep005 train=0.0851 val=0.1226 best=0.1192 bad=2
[TEACHER_B_FULLSEQ] ep010 train=0.0764 val=0.1191 best=0.1178 bad=3
[TEACHER_B_FULLSEQ] ep015 train=0.0770 val=0.1192 best=0.1176 bad=4
[TEACHER_B_FULLSEQ] ep020 train=0.0781 val=0.1170 best=0.1165 bad=2
[TEACHER_B_FULLSEQ] ep025 train=0.0718 val=0.1218 best=0.1165 bad=7
[TEACHER_B_FULLSEQ] ep030 train=0.0700 val=0.1172 best=0.1162 bad=1
[TEACHER_B_FULLSEQ] ep035 train=0.0711 val=0.1175 best=0.1157 bad=1
[TEACHER_B_FULLSEQ] ep040 train=0.0556 val=0.1152 best=0.1152 bad=0
[TEACHER_B_FULLSEQ] ep045 train=0.0519 val=0.1169 best=0.1152 bad=5
[TEACHER_B_FULLSEQ] ep050 train=0.0493 val=0.1200 best=0.1152 bad=10
Early stop.
✅ TEACHER_B_FULLSEQ guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/TEACHER_B_FULLSEQ.pt
⏱️ Tiempo entrenamiento B: 4719.2 s


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 15
# Modelo C: ventanas + fine-tuning por secuencia completa
# - Parte desde el modelo A
# ============================================
import os
import time
import copy
import numpy as np
import torch

TAG_C = "TEACHER_C_WIN_PLUS_FT"

LR_FT = 3e-4
EPOCHS_FT = 15
TBPTT_CHUNK_FT = 1024
lam_scale_C = 0.005
grad_clip_C = 1.0
patience_FT = 5

assert "model_A" in globals(), "Falta model_A. Corre antes la celda 13."

model_C = TeacherLSTM(
    in_dim=5,
    hid=HID_CMP,
    layers=LAY_CMP,
    dropout=DROPOUT_CMP,
    out_dim=3
).to(DEVICE)

model_C.load_state_dict(copy.deepcopy(model_A.state_dict()))
opt_C = torch.optim.AdamW(model_C.parameters(), lr=LR_FT)

best_val_C = float("inf")
best_state_C = None
bad = 0
t0_train = time.perf_counter()

for ep in range(1, EPOCHS_FT + 1):
    tr_losses = []
    for batch in dl_f_tr_cmp:
        tr_losses.append(
            train_one_full_event(
                model_C, opt_C,
                batch["X"], batch["Y"],
                tbptt_chunk=TBPTT_CHUNK_FT,
                train=True
            )
        )

    trL = float(np.mean(tr_losses)) if tr_losses else float("inf")

    va_losses = []
    for batch in dl_f_va_cmp:
        va_losses.append(
            train_one_full_event(
                model_C, opt_C,
                batch["X"], batch["Y"],
                tbptt_chunk=TBPTT_CHUNK_FT,
                train=False
            )
        )

    vaL = float(np.mean(va_losses)) if va_losses else float("inf")

    if vaL < best_val_C - 1e-6:
        best_val_C = vaL
        best_state_C = {k: v.detach().cpu().clone() for k, v in model_C.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if ep % 5 == 0 or ep == 1:
        print(f"[{TAG_C}] ep{ep:03d} train={trL:.4f} val={vaL:.4f} best={best_val_C:.4f} bad={bad}")

    if bad >= patience_FT:
        print("Early stop.")
        break

if best_state_C is not None:
    model_C.load_state_dict(best_state_C)

time_C = time.perf_counter() - t0_train

PT_C = os.path.join(RUNS_DIR_B, f"{TAG_C}.pt")
torch.save({
    "state_dict": model_C.state_dict(),
    "cfg": {
        "tag": TAG_C,
        "hid": HID_CMP,
        "layers": LAY_CMP,
        "dropout": DROPOUT_CMP,
        "lr_ft": LR_FT,
        "epochs_ft": ep,
        "train_mode": "windows_plus_fullseq_ft",
        "tbptt_chunk": TBPTT_CHUNK_FT,
        "base_init": PT_A if "PT_A" in globals() else "model_A_in_memory",
        "best_val": best_val_C,
        "XM": XM_CMP,
        "YM": YM_CMP,
        "X_P99ABS": X_P99ABS_CMP,
        "Y_P99ABS": Y_P99ABS_CMP
    }
}, PT_C)

print(f"✅ {TAG_C} guardado:", PT_C)
print(f"⏱️ Tiempo fine-tuning C: {time_C:.1f} s")
print(f"✅ Mejor val C: {best_val_C:.6f}")

[TEACHER_C_WIN_PLUS_FT] ep001 train=2.0026 val=1.7104 best=1.7104 bad=0
[TEACHER_C_WIN_PLUS_FT] ep005 train=1.2157 val=0.1650 best=0.1079 bad=1
Early stop.
✅ TEACHER_C_WIN_PLUS_FT guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/TEACHER_C_WIN_PLUS_FT.pt
⏱️ Tiempo fine-tuning C: 835.7 s
✅ Mejor val C: 0.107855


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 16
# Comparación final A/B/C sobre los 4 eventos test
# ============================================
import pandas as pd
import numpy as np
import time
import os

COMPARE_ROWS = []
PRED_STORE = {}

MODELS_CMP = {
    "A_windows": model_A,
    "B_fullseq": model_B,
    "C_win_ft": model_C,
}

TRAIN_TIMES_CMP = {
    "A_windows": time_A,
    "B_fullseq": time_B,
    "C_win_ft": time_A + time_C,
}

for model_name, model_obj in MODELS_CMP.items():
    for p in test_files_cmp:
        cache_item = cache_te_cmp[p]
        t0_eval = time.perf_counter()

        if model_name == "A_windows":
            rid, Yr, Yp, dt2 = predict_window_model_from_cache_cmp(
                model_obj, cache_item, stride_eval=16
            )
        else:
            rid, Yr, Yp, dt2 = predict_fullseq_model_from_cache_cmp(
                model_obj, cache_item, tbptt_chunk=1024
            )

        eval_time = time.perf_counter() - t0_eval

        i0, i1 = active_strict_indices_cmp(Yr, dt2, warmup_s=1.0, tail_margin_s=0.5)
        mets = regression_metrics_block_cmp(Yr, Yp, i0, i1)

        PRED_STORE[(model_name, rid)] = {
            "Yr": Yr,
            "Yp": Yp,
            "dt": dt2,
            "i0": i0,
            "i1": i1,
            "mets": mets
        }

        for row in mets:
            COMPARE_ROWS.append({
                "Model": model_name,
                "Event": rid,
                "dt_ref": dt2,
                "active_len": int(i1 - i0),
                "TrainTime_s": TRAIN_TIMES_CMP[model_name],
                "EvalTime_s": eval_time,
                **row
            })

df_compare = pd.DataFrame(COMPARE_ROWS)

print("✅ Comparación completa creada: df_compare")
display(df_compare)

print("\n=== PROMEDIO POR MODELO Y TARGET ===")
display(
    df_compare.groupby(["Model", "target"])[
        ["RMSE", "MAE", "Bias", "Corr", "PeakErrRel", "TrainTime_s", "EvalTime_s"]
    ].mean()
)

CSV_COMPARE = os.path.join(RUNS_DIR_B, "COMPARE_A_B_C_50_10_4.csv")
df_compare.to_csv(CSV_COMPARE, index=False)

print("✅ CSV guardado:", CSV_COMPARE)

✅ Comparación completa creada: df_compare


,Model,Event,dt_ref,active_len,TrainTime_s,EvalTime_s,target,RMSE,MAE,Bias,Corr,PeakTrueAbs,PeakPredAbs,PeakErrRel
0,A_windows,EQ006,0.005,11197,884.498340,36.407781,a_base_x,0.719902,0.487945,0.063928,0.272975,2.534030,4.428470,0.747600
1,A_windows,EQ006,0.005,11197,884.498340,36.407781,Fx,131.405435,97.001724,6.347548,0.332574,354.829010,793.689758,1.236823
2,A_windows,EQ006,0.005,11197,884.498340,36.407781,Mz,5325.982678,3967.464796,-368.979256,0.185881,2267.709961,16164.334961,6.128043
3,A_windows,EQ007,0.005,2812,884.498340,7.430173,a_base_x,0.159938,0.090661,0.051743,0.972851,2.683210,1.975263,0.263843
4,A_windows,EQ007,0.005,2812,884.498340,7.430173,Fx,23.024456,15.075431,11.057773,0.982051,371.343994,283.412354,0.236793
5,A_windows,EQ007,0.005,2812,884.498340,7.430173,Mz,589.340244,527.655698,-502.368505,0.768334,2458.649902,2662.480225,0.082903
6,A_windows,EQ019,0.005,10065,884.498340,35.477198,a_base_x,0.837621,0.365676,0.061473,0.924061,15.166000,11.885977,0.216275
7,A_windows,EQ019,0.005,10065,884.498340,35.477198,Fx,103.745299,49.360872,14.876746,0.955266,2517.760010,2237.688965,0.111238
8,A_windows,EQ019,0.005,10065,884.498340,35.477198,Mz,2751.671797,1659.557416,-710.222494,0.883432,24776.599609,19057.720703,0.230818
9,A_windows,EQ052,0.005,7852,884.498340,36.673176,a_base_x,0.082575,0.066841,0.057239,0.951805,0.888152,0.792783,0.107379



=== PROMEDIO POR MODELO Y TARGET ===


RMSE          MAE        Bias      Corr  \
Model     target                                                     
A_windows Fx          67.779518    43.077950   10.684804  0.812533   
          Mz        2309.731158  1663.958373 -519.733906  0.704960   
          a_base_x     0.450009     0.252781    0.058596  0.780423   
B_fullseq Fx          89.056609    57.814230   -3.041880  0.624692   
          Mz        1985.168185  1666.983547   35.017649  0.280207   
          a_base_x     0.652146     0.356887    0.016031  0.517804   
C_win_ft  Fx          85.759602    54.452817   -3.998353  0.704529   
          Mz        1936.879446  1621.416577   -2.055632  0.425883   
          a_base_x     0.611045     0.351533    0.019427  0.579742   

                    PeakErrRel  TrainTime_s  EvalTime_s  
Model     target                                         
A_windows Fx          0.414412   884.498340   28.997082  
          Mz          1.613005   884.498340   28.997082  
          a_base_x    0.333774   884.498340   28.997082  
B_fullseq Fx          0.243327  4719.241497    0.509664  
          Mz          0.267119  4719.241497    0.509664  
          a_base_x    0.454821  4719.241497    0.509664  
C_win_ft  Fx          0.313978  1720.189869    0.531094  
          Mz          0.479317  1720.189869    0.531094  
          a_base_x    0.370266  1720.189869    0.531094

✅ CSV guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/COMPARE_A_B_C_50_10_4.csv


In [ ]:
# ============================================
# NOTEBOOK B — CELDA 17
# Plots comparativos A/B/C
# - true vs A vs B vs C
# - guarda todo en un PDF
# ============================================
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os
import numpy as np

PDF_COMPARE = os.path.join(RUNS_DIR_B, "COMPARE_A_B_C_50_10_4_PLOTS.pdf")

with PdfPages(PDF_COMPARE) as pdf:
    # portada
    fig = plt.figure(figsize=(12, 3))
    plt.axis("off")
    plt.text(
        0.01, 0.7,
        "Comparación de estrategias de entrenamiento A/B/C\n"
        f"Eventos test: {TEST_IDS_CMP}",
        fontsize=12
    )
    plt.tight_layout()
    pdf.savefig(fig, dpi=180)
    plt.close(fig)

    for rid in TEST_IDS_CMP:
        for ch, ch_name in enumerate(Y_NAMES):
            fig = plt.figure(figsize=(12, 4))

            key_true = ("A_windows", rid)
            Yr = PRED_STORE[key_true]["Yr"]
            dt2 = PRED_STORE[key_true]["dt"]
            i0 = PRED_STORE[key_true]["i0"]
            i1 = PRED_STORE[key_true]["i1"]

            t = np.arange(Yr.shape[0]) * dt2
            sl = slice(i0, i1)

            plt.plot(t[sl], Yr[sl, ch], label="True", linewidth=2)

            for model_name in ["A_windows", "B_fullseq", "C_win_ft"]:
                Yp = PRED_STORE[(model_name, rid)]["Yp"]
                plt.plot(t[sl], Yp[sl, ch], label=model_name, alpha=0.9)

            plt.title(f"{rid} | {ch_name} | ACTIVE [{i0}:{i1}]")
            plt.xlabel("t [s]")
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()

            pdf.savefig(fig, dpi=180)
            plt.close(fig)

print("✅ PDF guardado:", PDF_COMPARE)

✅ PDF guardado: /content/memoria/runs_colab/runs_notebook_B_teacher/COMPARE_A_B_C_50_10_4_PLOTS.pdf
